缩小图片用LSD提取线段，看能否提取到骨干线段

In [ ]:
import os
from pytlsd import lsd
import cv2
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from datasets.dataset_reader import load_sparse_model
from datasets.line3dpp_loader import save_segments_l3dpp
from deeplsd.geometry.line_utils import clip_line_to_boundaries
from utils.visualize import viz_lines2D2
from deeplsd.geometry.viz_2d import save_plot, plot_image_with_lines


def resize_and_crop(image, size, interp_mode=None, crop = False):
    """ Apply a central crop to an image to resize it to a fixed size. """
    source_size = np.array(image.shape[:2], dtype=float)
    target_size = np.array(size, dtype=float)

    # Scale
    scale = np.amax(target_size / source_size)
    inter_size = np.round(source_size * scale).astype(int)
    if interp_mode is None:
        interp_mode = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    image = cv2.resize(image, (inter_size[1], inter_size[0]),
                       interpolation=interp_mode)

    # Central crop
    if crop:
        pad = np.round((source_size * scale - target_size) / 2.).astype(int)
        image = image[pad[0]:(pad[0] + int(target_size[0])),
                        pad[1]:(pad[1] + int(target_size[1]))]
    
    return image, inter_size



####################################### 参数 #######################################
workspace = r"/home/rylynn/Pictures/LinesDetection_Workspace/datasets/Dublin_block3"
resize = 1024

####################################### 路径 #######################################
sparse_model_path = os.path.join(workspace, 'sparse_txt')
images_path = os.path.join(workspace, 'test')
visualize_path = r"/home/rylynn/Pictures/LinesDetection_Workspace/output/Dublin_block3_lsd1024/visualize"
lines_path = r"/home/rylynn/Pictures/LinesDetection_Workspace/output/Dublin_block3_lsd1024/L3D++_data"
os.makedirs(visualize_path, exist_ok=True)
os.makedirs(lines_path, exist_ok=True)

# 0. 数据准备：读取稀疏模型
camerasInfo, points_in_images = load_sparse_model(sparse_model_path, image_scale=1)
print(f"[INFO] Loaded {len(camerasInfo)} images.")

for img_id in tqdm(range(len(camerasInfo))):
    cam_info = camerasInfo[img_id]
    width = int(cam_info['width'])
    height = int(cam_info['height'])
    img_name = cam_info['img_name'].split('/')[-1]
    img = cv2.imread(os.path.join(images_path, img_name+'.jpg'), cv2.IMREAD_GRAYSCALE)
    img,inter_size = resize_and_crop(img, (resize,resize))

    lines = lsd(img)[:,[0,1,2,3]].reshape(-1, 2, 2)
    lines, valid = clip_line_to_boundaries(lines, inter_size[[1,0]], min_len=0)
    lines = lines[valid]

    plot_image_with_lines(img, lines)
    save_plot(os.path.join(visualize_path, f"{os.path.splitext(img_name)[0]}.jpg"))
    plt.close()

    lines_matched_l3dpp = lines.reshape(-1,4)
    save_segments_l3dpp(lines_matched_l3dpp, lines_path, img_id+1, width, height)